In [5]:
import numpy as np
import pandas as pd
import itertools
import os
import matplotlib.pyplot as plt


In [ ]:
# CHANGE TO YOUR OWN LOCAL PATH TO THE WORKING DIRECTORY WHERE PREPPI, ZEPPI, DSCRIPT RESULTS ARE STORED.
localpath_prefix='/Users/userID/BayesianModel_forEcoliPPI/SupportingData'

#localpath_prefix='/Users/haizhao/Desktop/SupportingData'


In [7]:
#--- READ Ecoli-ZEPPI predictions; takes 3s~ after files are ready.

filetoread1=localpath_prefix+'/TestData_Ecoli-PPI-Interactomes/ecoli_ZEPPI_all.csv'

df_zp_ecl = pd.read_csv(filetoread1,sep=',',usecols=['Uni_pair','LR_SM','Zmaxmt'])

df_zp_ecl = df_zp_ecl.reset_index(drop=True)

print("Number of total_PPIs from ZEPPI:",len(df_zp_ecl))


Number of total_PPIs from ZEPPI: 5628400


In [8]:
#--- Read DS predictions; ~4s

filetoread2=localpath_prefix+'/TestData_Ecoli-PPI-Interactomes/ecoliK12_TT_all.tsv'

df_DS=pd.read_csv(filetoread2,sep='\t',names=["Uniprot1", "Uniprot2",'DS'])
df_DS = df_DS.dropna()
df_DS['Uni_pair']=df_DS['Uniprot1']+"_"+ df_DS['Uniprot2']
df_DS_ecl=df_DS[['Uni_pair','DS']]

print("Number of total_PPIs from TT(DS):",len(df_DS_ecl))


Number of total_PPIs from TT(DS): 9638235


In [ ]:
#--- Read PrePPI predictions (PrePPI_af), the non-structure ones; ~4s

def readPrePPIcsv_returncsv_sortedpair_score(filename,sep_symbol='\t',skipN=0):
    df0=pd.read_csv(filename,sep=sep_symbol,skiprows=skipN, usecols=[0,1,2])
    df0.columns=["UniprotID_A", "UniprotID_B","LR_SM"]
    df0['Uni_pair']= df0['UniprotID_A'].astype(str)+'_' +df0['UniprotID_B'].astype(str)
    return df0

def readdf_returndict_score(df0,lr_cut=0):
    if lr_cut==0:
        return  dict(zip(df0['Uni_pair'],df0['LR_SM']))
    else:
        df1 = df0[(df0['LR_SM'].astype(float) > int(lr_cut))]
        return dict(zip(df1['Uni_pair'],df1['LR_SM']))

pred_af_lr0_nonstr_df = readPrePPIcsv_returncsv_sortedpair_score(localpath_prefix+'/TestData_Ecoli-PPI-Interactomes/ecoli_PrePPI_SM_ns.csv', sep_symbol=',')
pred_af_lr0_nonstr_dict = readdf_returndict_score(pred_af_lr0_nonstr_df,lr_cut=0);

print("Number of total_PPIs from PrePPI_nonstr:",len(pred_af_lr0_nonstr_df))


Number of total_PPIs from PrePPI_nonstr: 5816242


In [50]:
import sys, os
sys.path.append(os.path.abspath("../src"))

from Bayes import DiscretizedNaiveBayes_IndependentFeatures
from Bayes import DiscretizedNaiveBayes_JointBins



In [51]:
# Import Bayes models that were trained on Human

import pickle
with open('../data/human_ZP_Bayes_ratio1000.pkl', 'rb') as f:
    nbn_ZP = pickle.load(f)

with open('../data/human_DS_Bayes_STRING.pkl', 'rb') as f:
    nbn_DS = pickle.load(f)


In [52]:
# Apply the trained Bayes model to E. coli; takes ~1min

df_zp_ecl['LR_ZP'] = nbn_ZP.predict(df_zp_ecl['Zmaxmt'].to_numpy().reshape(len(df_zp_ecl),1))
df_DS_ecl['LR_DS'] = nbn_DS.predict(df_DS_ecl['DS'].to_numpy().reshape(len(df_DS_ecl),1))



/var/folders/g9/763yp4f55l12h76s_v2530480000gq/T/ipykernel_34368/1376985900.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_DS_ecl['LR_DS'] = nbn_DS.predict(df_DS_ecl['DS'].to_numpy().reshape(len(df_DS_ecl),1))


In [53]:
#--- Combine three different clues; fill the missing LR as 1; ~40s

# Merge all three DataFrames
df_threeclues = pred_af_lr0_nonstr_df.merge(df_DS_ecl, on='Uni_pair', how='outer').merge(df_zp_ecl, on='Uni_pair', how='outer')

# Fill NaN values with 1
df_threeclues.fillna({'LR_SM':1}, inplace=True)
df_threeclues.fillna({'LR_DS':1}, inplace=True)
df_threeclues.fillna({'LR_ZP':1}, inplace=True)

# Compute the total score
df_threeclues['LR_INT'] = df_threeclues['LR_SM'] * df_threeclues['LR_DS'] * df_threeclues['LR_ZP']

print(len(df_threeclues))


9642046


In [ ]:
#--- Write the output to a CSV file in the current directory; takes 23s

#df_threeclues_out=df_threeclues[['Uni_pair','LR_SM','DS','LR','Zmaxmt','LR_ZP', 'LR_DS','LR_total', 'LR_SMDS', 'LR_ZPDS', 'LR_SMZP']]

df_threeclues_out=df_threeclues[['Uni_pair','LR_SM','LR_ZP','LR_DS','LR_INT']]

fileTOwrite=localpath_prefix+'/ecoli_PPIs_LRINT_threeclues.csv'

df_threeclues_out.to_csv(fileTOwrite,index=False)
